| 概念 | 問い | DeclareSort の標準 |
| --- | --- | --- |
| DCA（領域閉包仮定） | 名前を挙げた個体が全個体か？ | 採用しない |
| UNA（異名仮定） | 異なる名前は異なる個体か？ | 採用しない |

## 独自Sortを持つFOLの関数

In [1]:
from typing import cast

from z3 import (
    BoolSort,
    Const,
    DeclareSort,
    ExprRef,
    FuncDeclRef,
    Function,
    ModelRef,
    Solver,
    SortRef,
    sat,
)

# 独自Sort
Object: SortRef = DeclareSort("Object")
Region: SortRef = DeclareSort("Region")
Class: SortRef = DeclareSort("Class")

# 定数記号
cup: ExprRef = cast(ExprRef, Const("cup", Object))
mug: ExprRef = cast(ExprRef, Const("mug", Object))

kitchen: ExprRef = cast(ExprRef, Const("kitchen", Region))
shelf: ExprRef = cast(ExprRef, Const("shelf", Region))

Container: ExprRef = cast(ExprRef, Const("Container", Class))
MovableObject: ExprRef = cast(ExprRef, Const("MovableObject", Class))

# 関数・述語記号
location: FuncDeclRef = Function("location", Object, Region)
hasType: FuncDeclRef = Function("hasType", Object, Class, BoolSort())

s: Solver = Solver()

s.add(
    location(cup) == kitchen,
    location(mug) == shelf,
    kitchen != shelf,
    hasType(cup, Container),
    hasType(cup, MovableObject),
)

print(s.check())

if s.check() == sat:
    m: ModelRef = s.model()
    print("location(cup):", m.evaluate(location(cup)))
    print("hasType(cup, Container):", m.evaluate(hasType(cup, Container)))


sat
location(cup): Region!val!0
hasType(cup, Container): True


## 合同性を試す

In [2]:
s.push()

s.add(cup == mug)

print(s.check())  # unsat

s.pop()

unsat


## UNSAT Core

In [4]:
from z3 import (
    Bool,
    BoolRef,
    Const,
    DeclareSort,
    ExprRef,
    FuncDeclRef,
    Function,
    Solver,
    SortRef,
    unsat,
)

Object: SortRef = DeclareSort("Object")
Region: SortRef = DeclareSort("Region")

cup: ExprRef = Const("cup", Object)
mug: ExprRef = Const("mug", Object)
other: ExprRef = Const("other", Object)

kitchen: ExprRef = Const("kitchen", Region)
shelf: ExprRef = Const("shelf", Region)

location: FuncDeclRef = Function("location", Object, Region)

s: Solver = Solver()

# 各Assertionを識別子付きで登録
s.assert_and_track(location(cup) == kitchen, "a1")
s.assert_and_track(cup == mug, "a2")

# 結論に無関係な情報
s.assert_and_track(location(other) == shelf, "a3")
s.assert_and_track(kitchen != shelf, "a4")
s.assert_and_track(other != cup, "a5")

# 前提集合が整合していることを確認
print("Consistency:", s.check())

# 結論の否定を一時的に追加
s.push()
s.add(location(mug) != kitchen)

result = s.check()
print("Entailment test:", result)

if result == unsat:
    core: list[str] = [str(a) for a in s.unsat_core()]
    print("Support candidate:", core)

s.pop()


Consistency: sat
Entailment test: unsat
Support candidate: ['a1', 'a2']
